## Importing necessary libraries and functions

In [1]:
import numpy as np
from tomograph_functions import radon_transform,back_projection,filtr_sinogram
from helpers import normalize_img,read_img,show_img
import os
import ipywidgets.widgets as widgets
import ipywidgets
from IPython.display import display
import warnings
#Supressing warrnigns
warnings.filterwarnings("ignore")

## Selecting the input image and set options of simulation

In [20]:
available_images=[file.name for file in os.scandir("./tomograf-obrazy") if file.is_file()]

detectors=widgets.IntSlider(
    value=180,
    min=2,
    max=360,
    step=1,
    description='Detectors:',
    disabled=False,
    continuous_update=False,
    orientation='horizontal',
    readout=True,
    readout_format='d',
    layout=widgets.Layout(width='auto')
)
detectors_spread=widgets.FloatSlider(
    value=90.0,
    min=45.0,
    max=270,
    step=0.5,
    description='Angular spread:',
    disabled=False,
    continuous_update=False,
    orientation='horizontal',
    readout=True,
    readout_format='.1f',
    layout=widgets.Layout(width='auto')
)

step=widgets.FloatSlider(
    value=1.0,
    min=0.1,
    max=10.0,
    step=0.1,
    description='Step value:',
    disabled=False,
    continuous_update=False,
    orientation='horizontal',
    readout=True,
    readout_format='.1f',
    layout=widgets.Layout(width='auto')
)

cbox=widgets.Combobox(

    placeholder='Choose an image',
    options=available_images,
    description='Combobox:',
    ensure_option=True,
    disabled=False,
    layout=widgets.Layout(width='auto')
)
filtering=widgets.Checkbox(
    value=True,
    description='Use filtering',
    disabled=False,
    layout=widgets.Layout(width='auto')
)
show_steps=widgets.Checkbox(
    value=False,
    description='Show steps',
    disabled=False,
    layout=widgets.Layout(width='auto')
)
sliders=widgets.VBox([detectors,detectors_spread,step,cbox,filtering,show_steps],layout=widgets.Layout(width='45%'))
display(sliders)

In [ ]:

import ipywidgets as widgets
from IPython.display import display

if cbox.value != "":
    image = read_img(f"tomograf-obrazy/{cbox.value}")
    height, width = image.shape

    intermediate_sinograms, sinogram = radon_transform(
        image, detectors.value, step.value, detectors_spread.value
    )

    filtered_str = ""
    if filtering.value:
        sinogram = filtr_sinogram(sinogram)
        filtered_str = " (with filtering)"
        intermediate_sinograms = [filtr_sinogram(part_sin) for part_sin in intermediate_sinograms]

    intermediate_reconstructed, final = back_projection(
        sinogram, height, width, detectors.value, step.value, detectors_spread.value
    )
    final=normalize_img(final)
    intermediate_reconstructed=[normalize_img(recon) for recon in intermediate_reconstructed]

    input_image_out = widgets.Output()
    with input_image_out:
        show_img(image, title="Input image")

    final_sinogram_out = widgets.Output()
    with final_sinogram_out:
        show_img(sinogram, title=f"Sinogram{filtered_str}")

    final_image_out = widgets.Output()
    with final_image_out:
        show_img(final, title=f"Final image{filtered_str}")

    images_box = widgets.HBox([input_image_out, final_sinogram_out, final_image_out])
    display(images_box)
    

In [24]:
#Wrappers for show_img to make interactive dispaying
if show_steps.value:
    def show_sinogram(step: int) -> None:
        show_img(intermediate_sinograms[step], title=f"Sinogram {filtered_str} number {step}")

    def show_reconstructed(step:int)->None:
        show_img(intermediate_reconstructed[step], title=f"Reconstructed {filtered_str} number {step}")

    slider_sinogram = widgets.IntSlider(
        min=0,
        max=len(intermediate_sinograms) - 1,
        step=1,
        value=0
        #interval=100
    )

    slider_reconstructed = widgets.IntSlider(#widgets.Play(
        min=0,
        max=len(intermediate_reconstructed) - 1,
        step=1,
        value=0,
        interval=100
    )

    part_sinogram = widgets.interactive_output(show_sinogram, {'step': slider_sinogram})
    part_reconstructed=widgets.interactive_output(show_reconstructed,{'step': slider_reconstructed})


    sliders = widgets.HBox([slider_sinogram,slider_reconstructed])
    part_images=widgets.HBox([part_sinogram,part_reconstructed])
    combined=widgets.VBox([sliders,part_images])
    display(combined)